# Metadata gaps — three figures

1. Evolution per century of the share of individuals **without** a Cliopatria polity (among those with a floruit period).
2. The 20 polities with the **worst** floruit coverage (≥ 8 000 attributed individuals).
3. The 20 polities with the **best** floruit coverage (≥ 8 000 attributed individuals).

Individuals born after 1995 who are still living are excluded from the polity-floruit calculations (their floruit period has not yet been established and would inflate the missing-data share).

In [ ]:
import duckdb
import numpy as np
import polars as pl
import matplotlib as mpl
import matplotlib.pyplot as plt

DB_PATH = "../data/humans_clean.duckdb"
TOO_YOUNG_CUTOFF = 1995
MIN_N_EXTREMES = 8000

mpl.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})
COLOR_BAD = "#b5542a"
COLOR_GOOD = "#3a8a3a"
COLOR_LINE = "#2171b5"

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)

flo = conn.execute("""
    SELECT wikidata_id, birth_year, death_year,
           floruit_period_start
    FROM individuals_floruit_period
""").pl()
clio_ids_df = conn.execute("SELECT DISTINCT wikidata_id FROM individuals_cliopatria").pl()
clio_ids = set(clio_ids_df['wikidata_id'].to_list())
clio = conn.execute("SELECT wikidata_id, polity_id FROM individuals_cliopatria").pl()
polities = conn.execute("SELECT id, name FROM polities_cliopatria").pl()
conn.close()

flo = flo.with_columns(
    pl.col('floruit_period_start').is_not_null().alias('has_floruit'),
    pl.col('wikidata_id').is_in(clio_ids).alias('has_polity'),
    (pl.col('birth_year').is_not_null()
     & (pl.col('birth_year') > TOO_YOUNG_CUTOFF)
     & pl.col('death_year').is_null()).alias('too_young'),
)
n_raw = flo.height
flo = flo.filter(~pl.col('too_young'))
print(f"individuals (raw):              {n_raw:,}")
print(f"excluded (born after {TOO_YOUNG_CUTOFF}, alive): {n_raw - flo.height:,}")
print(f"individuals (analysis cohort):  {flo.height:,}")
print(f"  with floruit period:          {int(flo['has_floruit'].sum()):,}")
print(f"  matched to a polity:          {int(flo['has_polity'].sum()):,}")

## Figure 1 — % of individuals **without** a polity, per century

Among individuals with a floruit period, the share **not** matched to any Cliopatria polity, by century of `floruit_period_start`.

In [ ]:
f = (
    flo.filter(pl.col('has_floruit'))
    .with_columns(((pl.col('floruit_period_start') // 100) * 100).alias('century'))
    .filter(pl.col('century').is_between(-3000, 2000))
)

rate = (
    f.group_by('century')
    .agg([
        pl.len().alias('n'),
        pl.col('has_polity').sum().alias('matched'),
    ])
    .with_columns(((1 - pl.col('matched') / pl.col('n')) * 100).alias('no_polity_pct'))
    .sort('century')
)
rate_plot = rate.filter(pl.col('n') >= 50)

fig, ax = plt.subplots(figsize=(11, 4.5))
xs = rate_plot['century'].to_list()
ys = rate_plot['no_polity_pct'].to_list()
ax.fill_between(xs, 0, ys, color=COLOR_BAD, alpha=0.18, lw=0)
ax.plot(xs, ys, color=COLOR_BAD, lw=1.6, marker="o", ms=3)
ax.axvspan(400, 800, color="#888", alpha=0.08, lw=0)
ax.axvspan(1700, 2000, color="#888", alpha=0.08, lw=0)
ax.text(600, 95, "early medieval", ha="center", fontsize=9, color="#555")
ax.text(1850, 95, "modern", ha="center", fontsize=9, color="#555")
ax.axhline(50, color="#888", lw=0.7, ls=":")
ax.set_ylim(0, 100)
ax.set_xlabel("century of floruit_period_start")
ax.set_ylabel("% without polity match")
ax.set_title("Individuals without a Cliopatria polity, by century\n"
             "(among those with a floruit period)")
ax.grid(axis="y", alpha=0.2, ls="--")
plt.tight_layout()
plt.show()

### Per-polity floruit-coverage table

Used by figures 2 and 3.

In [ ]:
name_map = dict(polities.select(['id', 'name']).iter_rows())

elig_ids = set(flo['wikidata_id'].to_list())
clio_filt = clio.filter(pl.col('wikidata_id').is_in(elig_ids))
ce = (
    clio_filt.with_columns(pl.col('polity_id').str.split(';').alias('polity_int'))
    .explode('polity_int')
    .with_columns(pl.col('polity_int').cast(pl.Int64, strict=False))
    .drop_nulls('polity_int')
)

floruit_ids = set(flo.filter(pl.col('has_floruit'))['wikidata_id'].to_list())
ce = ce.with_columns(pl.col('wikidata_id').is_in(floruit_ids).alias('has_floruit'))

polity_floruit = (
    ce.group_by('polity_int')
    .agg([
        pl.len().alias('n'),
        pl.col('has_floruit').sum().alias('n_floruit'),
    ])
    .with_columns(
        (pl.col('n_floruit') / pl.col('n') * 100).alias('share_floruit'),
        pl.col('polity_int').replace_strict(name_map, default=None).alias('polity'),
    )
)

eligible = polity_floruit.filter(pl.col('n') >= MIN_N_EXTREMES)
print(f"{eligible.height} polities with n >= {MIN_N_EXTREMES:,} attributed individuals")

## Figure 2 — 20 polities with the **worst** floruit coverage  (n ≥ 8 000)

In [ ]:
worst20 = (
    eligible.with_columns((100 - pl.col('share_floruit')).alias('pct_missing'))
    .sort('pct_missing', descending=True)
    .head(20)
    .reverse()
)

fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(worst20.height)
pct_missing = worst20['pct_missing'].to_numpy()
ax.barh(y, pct_missing, color=COLOR_BAD,
        edgecolor="white", lw=0.6, height=0.75)
for yi, r in zip(y, worst20.iter_rows(named=True)):
    ax.text(r['pct_missing'] + 0.8, yi,
            f"{r['pct_missing']:.0f}%  (n={int(r['n']):,})",
            va="center", ha="left", fontsize=9, color="#444")
ax.set_yticks(y)
ax.set_yticklabels(worst20['polity'].to_list())
ax.set_xlim(0, 118)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("% of attributed individuals with no floruit period")
ax.set_title(f"20 polities with the worst floruit coverage  (n >= {MIN_N_EXTREMES:,})")
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

## Figure 3 — 20 polities with the **best** floruit coverage  (n ≥ 8 000)

In [ ]:
best20 = (
    eligible.sort('share_floruit', descending=True)
    .head(20)
    .reverse()
)

fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(best20.height)
share_floruit_arr = best20['share_floruit'].to_numpy()
ax.barh(y, share_floruit_arr, color=COLOR_GOOD,
        edgecolor="white", lw=0.6, height=0.75)
for yi, r in zip(y, best20.iter_rows(named=True)):
    ax.text(r['share_floruit'] + 0.8, yi,
            f"{r['share_floruit']:.0f}%  (n={int(r['n']):,})",
            va="center", ha="left", fontsize=9, color="#444")
ax.set_yticks(y)
ax.set_yticklabels(best20['polity'].to_list())
ax.set_xlim(0, 118)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("% of attributed individuals with a floruit period")
ax.set_title(f"20 polities with the best floruit coverage  (n >= {MIN_N_EXTREMES:,})")
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()